### "Statistics and Data Analysis" 32: Monte Carlo Sampling and Bootstrapping in Bayesian Inference
In this lecture, Dr. Brunton explains Monte Carlo sampling and bootstrapping in Bayesian inference using both theoretical explanations and Python code examples. This lecture details a numerical method for approximating posterior distributions, particularly useful when traditional methods are insufficient. Learn how sampling, weighting, and resampling techniques provide estimates even for complex scenarios. 

https://www.youtube.com/watch?v=f1Vcc-bPfnU&list=PLMrJAkhIeNNT14qn1c5qdL29A1UaHamjx&index=32

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom, uniform
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# True value of theta (probability of heads)
theta_true = 0.5

# Number of coin flips per batch and total flips
batch_size = 10
total_flips = 250

# Generate the true coin flip outcomes based on the true theta
np.random.seed(42)  # For reproducibility
coin_flips = np.random.binomial(1, theta_true, total_flips)  # 1 for heads, 0 for tails

# Number of Monte Carlo samples
num_samples = 5000

# Prior: Uniform distribution for theta (equivalent to Beta(1, 1))
prior_thetas = np.random.uniform(0, 1, num_samples)

# Set up the figure with two subplots: one for the density and one for the mean
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))

# Adjust the space between the subplots
plt.subplots_adjust(hspace=0.3)

# Set up the first plot for the sampled posterior distribution
theta_range = np.linspace(0, 1, 100)

line1, = ax1.plot([], [], lw=2)
mean_line, = ax1.plot([], [], lw=2, linestyle='dashed', color='orange')  # Dashed line for mean
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 10)
ax1.set_title('Monte Carlo Posterior Sampling Evolution')
ax1.set_xlabel('Theta (probability of heads)')
ax1.set_ylabel('Density')

# Set up the second plot for mean evolution
line2, = ax2.plot([], [], lw=2, color='orange')
ax2.set_xlim(0, total_flips // batch_size)
ax2.set_ylim(0, 1)
ax2.set_title('Evolution of Posterior Mean')
ax2.set_xlabel('Batch Number')
ax2.set_ylabel('Posterior Mean')

# Initialize shaded regions for both plots
std_shade1 = ax1.fill_between([], [], [], color='orange', alpha=0.2)
std_shade2 = ax2.fill_between([], [], [], color='orange', alpha=0.2)

# To store the evolution of the mean and std after each batch
posterior_means = []
posterior_stds = []
batch_numbers = []

# Function to initialize the plots
def init():
    line1.set_data([], [])
    mean_line.set_data([], [])
    return line1, mean_line, line2

# Function to update the plots after each batch of flips
def update(batch_number):
    global prior_thetas, std_shade1, std_shade2
    
    # Get the current batch of flips
    start = batch_number * batch_size
    end = start + batch_size
    batch_flips = coin_flips[start:end]
    
    # Calculate the number of heads and tails in the current batch
    heads = np.sum(batch_flips)
    tails = batch_size - heads
    
    # Likelihood for each prior sample given the data (Binomial likelihood)
    likelihoods = binom.pmf(heads, batch_size, prior_thetas)
    
    # Reweight the prior thetas based on the likelihood
    weights = likelihoods / np.sum(likelihoods)
    
    # Resample prior thetas according to the weights to approximate the posterior
    posterior_thetas = np.random.choice(prior_thetas, size=num_samples, p=weights)
    
    # Update the prior thetas for the next iteration (Monte Carlo approximation)
    prior_thetas = posterior_thetas
    
    # Calculate the posterior mean and standard deviation
    posterior_mean = np.mean(posterior_thetas)
    posterior_std = np.std(posterior_thetas)
    posterior_means.append(posterior_mean)
    posterior_stds.append(posterior_std)
    batch_numbers.append(batch_number)
    
    # Update the line for the sampled posterior distribution (density plot)
    density, _ = np.histogram(posterior_thetas, bins=theta_range, density=True)
    line1.set_data(theta_range[:-1], density)
    
    # Dynamically update the y-axis limit based on the max value of the density
    ax1.set_ylim(0, 1.1 * np.max(density))
    
    # Update the line for the evolution of the mean
    line2.set_data(batch_numbers, posterior_means)
    
    # Remove the previous shaded region (properly clearing it)
    for coll in ax1.collections:
        coll.remove()
    for coll in ax2.collections:
        coll.remove()
    
    # Update the shaded region for the posterior distribution plot (±1 std deviation)
    std_shade1 = ax1.fill_between(
        theta_range[:-1],
        density*(theta_range[:-1] >= posterior_mean - posterior_std) * 
        (theta_range[:-1] <= posterior_mean + posterior_std),
        color='orange',
        alpha=0.4
    )
    
    # Update the shaded region for the mean plot (±1 std deviation)
    std_shade2 = ax2.fill_between(
        batch_numbers,
        np.array(posterior_means) - np.array(posterior_stds),
        np.array(posterior_means) + np.array(posterior_stds),
        color='orange',
        alpha=0.4
    )
    
    # Update the vertical dashed line for the mean in the posterior plot
    mean_line.set_data([posterior_mean, posterior_mean], [0, np.max(density)])  # Dashed line from y=0 to max density
    
    # Update titles
    ax1.set_title(f'Batch {batch_number + 1}: Posterior after {end} flips')
    
    return line1, mean_line, line2

# Create the animation
ani = FuncAnimation(fig, update, frames=total_flips // batch_size, init_func=init, blit=True, repeat=False)

# Display the animation
HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
num_samples = 5000
prior_thetas = np.random.uniform(0, 1, num_samples)

plt.hist(prior_thetas, bins=50, density=True, alpha=0.7, color='blue')
plt.title('Initial Prior Distribution (Uniform)')
plt.xlabel('Theta')
plt.ylabel('Density')
plt.axhline(1.0, color='red', linestyle='--', label='Theoretical Density (1.0)')
plt.legend()
plt.show()

In [ ]:

print(binom.pmf([1,2,3,4,5,6,7,8,9,10], 10, 0.5))
print(binom.pmf(5, 10, [0.1*x for x in range(10)]))  # Likelihood of getting 5 heads in 10 flips with theta=0.5